In [2]:
import pandas as pd
import plotly.express as px

df = pd.read_csv('../sp500_companies.csv')
df.head()

,Exchange,Symbol,Shortname,Longname,Sector,Industry,Currentprice,Marketcap,Ebitda,Revenuegrowth,City,State,Country,Fulltimeemployees,Longbusinesssummary,Weight
0,NMS,AAPL,Apple Inc.,Apple Inc.,Technology,Consumer Electronics,254.49,3846819807232,1.346610e+11,0.061,Cupertino,CA,United States,164000.0,"Apple Inc. designs, manufactures, and markets ...",0.069209
1,NMS,NVDA,NVIDIA Corporation,NVIDIA Corporation,Technology,Semiconductors,134.70,3298803056640,6.118400e+10,1.224,Santa Clara,CA,United States,29600.0,NVIDIA Corporation provides graphics and compu...,0.059350
2,NMS,MSFT,Microsoft Corporation,Microsoft Corporation,Technology,Software - Infrastructure,436.60,3246068596736,1.365520e+11,0.160,Redmond,WA,United States,228000.0,Microsoft Corporation develops and supports so...,0.058401
3,NMS,AMZN,"Amazon.com, Inc.","Amazon.com, Inc.",Consumer Cyclical,Internet Retail,224.92,2365033807872,1.115830e+11,0.110,Seattle,WA,United States,1551000.0,"Amazon.com, Inc. engages in the retail sale of...",0.042550
4,NMS,GOOGL,Alphabet Inc.,Alphabet Inc.,Communication Services,Internet Content & Information,191.41,2351625142272,1.234700e+11,0.151,Mountain View,CA,United States,181269.0,Alphabet Inc. offers various products and plat...,0.042309


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 502 entries, 0 to 501
Data columns (total 16 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Exchange             502 non-null    object 
 1   Symbol               502 non-null    object 
 2   Shortname            502 non-null    object 
 3   Longname             502 non-null    object 
 4   Sector               502 non-null    object 
 5   Industry             502 non-null    object 
 6   Currentprice         502 non-null    float64
 7   Marketcap            502 non-null    int64  
 8   Ebitda               473 non-null    float64
 9   Revenuegrowth        499 non-null    float64
 10  City                 502 non-null    object 
 11  State                482 non-null    object 
 12  Country              502 non-null    object 
 13  Fulltimeemployees    493 non-null    float64
 14  Longbusinesssummary  502 non-null    object 
 15  Weight               502 non-null    flo

In [7]:
df['Marketcap_B'] = df['Marketcap'] / 1e9

fig = px.histogram(df, x='Marketcap_B', nbins=50, title='Distribuição de Market Cap (em bilhões USD)')
fig.show()

### Distribuição de Market Cap (Valor de Mercado)

- A enorme maioria das 500 empresas tem market cap abaixo de ~200B
- Existem pouquíssimas empresas com market cap acima de 1000B (1T) e chegam até ~3500B (3.5T)
- O gráfico mostra que o S&P 500 não é equilibrado: é dominado por um punhado de gigantes enquanto a maior parte das empresas é relativamente pequena

In [6]:
df_filtered = df[df['Marketcap_B'] < 500]

fig = px.histogram(df_filtered, x='Marketcap_B', nbins=50,
                   title='Distribuição de Market Cap - empresas abaixo de US$500B')
fig.show()

### Distribuição de Market Cap (Valor de mercado) - Empresas abaixo de US$500B

- Mesmo dentro das empresas menores, a concentração continua: a maioria está abaixo de 100B
- Empresas entre 100B e 500B já são consideradas grandes, mas são minoria mesmo nesse grupo filtrado
- O histograma revela que "entrar no S&P 500" não significa ser grande, pois a maioria das empresas do índice tem tamanho modesto comparado às líderes

### Market Cap vs Crescimento de Receita

Empresas maiores crescem menos? Exploramos a relação entre 
tamanho e crescimento de receita por setor.

In [11]:
color_map = {
    'Technology': '#00CC96',
    'Real Estate': '#1f1f8a',
    'Consumer Cyclical': '#EF553B',
    'Communication Services': '#AB63FA',
    'Financial Services': '#FFA15A',
    'Consumer Defensive': '#19D3F3',
    'Healthcare': '#FF6692',
    'Energy': '#B6E880',
    'Basic Materials': '#FF97FF',
    'Industrials': '#FECB52',
    'Utilities': '#636EFA'
}

fig = px.scatter(df_scatter,
                 x='Marketcap_B',
                 y='Revenuegrowth',
                 color='Sector',
                 color_discrete_map=color_map,
                 hover_name='Shortname',
                 title='Market Cap vs Crescimento de Receita por Setor',
                 labels={'Marketcap_B': 'Market Cap (USD Bilhões)',
                         'Revenuegrowth': 'Crescimento de Receita'})
fig.show()

 ### Conclusões
 
 1. Empresas maiores crescem menos
 
As megacaps (acima de 1000B) estão todas com crescimento de receita entre -0.1 e 0.3 (-10 a 30%) ano a ano. O ponto isolado em ~3300B com crescimento de 1.2 (120%) é provavelmente a Nvidia, que é uma exceção notável.

2. Alto crescimento está concentrado em empresas menores

Todos os pontos com crescimento acima de 0.5 estão à esquerda, abaixo de 200B de market cap. Empresas pequenas têm mais espaço pra crescer.

3. Crescimento negativo também existe

Algumas empresas têm revenue growth negativo (abaixo de 0), espalhadas por vários setores: não é exclusivo de um setor específico.

4. A maioria das empresas cresce de forma modesta

A concentração de pontos entre 0 e 0.3 de crescimento mostra que crescimento de receita acima de 30% é raro no S&P 500, o que faz sentido pois são empresas maduras.   

## Crescimento de Receita por Setor

Quais setores do S&P 500 estão crescendo mais? 
Calculamos a média de crescimento de receita anual por setor.

In [12]:
df_growth = df.dropna(subset=['Revenuegrowth', 'Sector'])
growth_by_sector = df_growth.groupby('Sector')['Revenuegrowth'].mean().reset_index()
growth_by_sector = growth_by_sector.sort_values('Revenuegrowth', ascending=False)

fig = px.bar(growth_by_sector,
             x='Sector',
             y='Revenuegrowth',
             title='Crescimento Médio de Receita por Setor',
             labels={'Revenuegrowth': 'Crescimento Médio de Receita', 'Sector': 'Setor'},
             color='Revenuegrowth',
             color_continuous_scale='RdYlGn')
fig.update_layout(xaxis_tickangle=-45)
fig.show()

### Conclusões:

1. Financial Services e Technology lideram o crescimento

Os dois setores com maior crescimento médio de receita (~12-13%), bem acima dos demais. Financial Services na liderança é uma surpresa, pois normalmente tecnologia domina essa métrica.

2. Existe uma divisão clara entre setores

Há um grupo de alto crescimento (Financial Services, Technology, Real Estate, Healthcare acima de 7%) e um grupo de baixo crescimento (Industrials, Utilities, Basic Materials, Consumer Defensive, Energy abaixo de 5%). O S&P 500 não cresce de forma homogênea.

3. Energy e Consumer Defensive crescem menos

Os setores mais "defensivos" e maduros aparecem no fim da lista — faz sentido, são setores estáveis mas sem grande expansão de receita.

4. Real Estate surpreende

Empresas do ramo imobiliário aparecem em terceiro lugar, acima de Healthcare e Consumer Cyclical, o que é incomum e vale investigar se é um dado pontual ou uma tendência.

## Lucratividade por Setor

Analisamos lucratividade por dois ângulos:
- **Volume**: EBITDA médio absoluto, ou seja, quais setores geram mais lucro em tamanho
- **Valuation**: Market Cap/EBITDA, ou seja, quantas vezes o mercado paga pelo lucro de cada setor (quanto maior, mais "caro/sobrevalorizado" o setor)

In [17]:
df_ebitda = df.dropna(subset=['Ebitda', 'Sector', 'Marketcap'])
df_ebitda['Ebitda_B'] = df_ebitda['Ebitda'] / 1e9

# Gráfico 1: EBITDA absoluto
ebitda_by_sector = df_ebitda.groupby('Sector')['Ebitda_B'].mean().reset_index()
ebitda_by_sector = ebitda_by_sector.sort_values('Ebitda_B', ascending=False)

fig1 = px.bar(ebitda_by_sector,
              x='Sector',
              y='Ebitda_B',
              title='EBITDA Médio por Setor (USD Bilhões)',
              labels={'Ebitda_B': 'EBITDA Médio (USD Bilhões)', 'Sector': 'Setor'},
              color='Ebitda_B',
              color_continuous_scale='RdYlGn')
fig1.update_layout(xaxis_tickangle=-45)
fig1.show()



C:\Users\Caio Pontes\AppData\Local\Temp\ipykernel_12444\1840831337.py:2: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



### Concluões:

1. Communication Services domina em lucro absoluto (~25B) puxado por gigantes como Google e Meta
2. Energy surpreende em segundo lugar, acima de Technology em volume bruto
3. Real Estate, apesar de apresentar alto crescimento, é o menos lucrativo em tamanho

In [18]:
# Gráfico 2: Marketcap/EBITDA
df_ebitda_filtered = df_ebitda[df_ebitda['Sector'] != 'Financial Services']
df_ebitda_filtered['Valuation_Multiple'] = df_ebitda_filtered['Marketcap'] / df_ebitda_filtered['Ebitda']

multiple_by_sector = df_ebitda_filtered.groupby('Sector')['Valuation_Multiple'].mean().reset_index()
multiple_by_sector = multiple_by_sector.sort_values('Valuation_Multiple', ascending=False)

fig2 = px.bar(multiple_by_sector,
              x='Sector',
              y='Valuation_Multiple',
              title='Valuation: Market Cap/EBITDA por Setor (excl. Financial Services)',
              labels={'Valuation_Multiple': 'Market Cap/EBITDA (múltiplo)', 'Sector': 'Setor'},
              color='Valuation_Multiple',
              color_continuous_scale='RdYlGn_r')
fig2.update_layout(xaxis_tickangle=-45)
fig2.show()

C:\Users\Caio Pontes\AppData\Local\Temp\ipykernel_12444\2349006238.py:3: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



## Conclusões:

Technology é o setor mais caro do índice, com múltiplo de ~37x: o mercado paga 37 vezes o EBITDA (lucro) para investir em tech.

Energy e Utilities são os mais baratos, em torno de 8-9x.
Existe uma divisão clara: setores de crescimento (tech, comunicação, saúde) com múltiplos altos vs. setores maduros (energia, utilities, materiais) com múltiplos baixos

OBS: Financial Services foi excluído da análise de valuation porque o EBITDA não é uma métrica adequada para bancos e seguradoras. Diferente de empresas industriais ou de tecnologia, instituições financeiras  têm "juros" como parte central do seu modelo de negócio, e o EBITDA ignora  juros por definição. Por isso o múltiplo aparece distorcido (negativo) para esse setor, 
não refletindo a realidade. 

## Desigualdade por Setor

O S&P 500 não é homogêneo dentro de cada setor. 
Usamos um box plot para revelar a dispersão de market cap dentro de cada setor.
Quanto maior a caixa, mais desigual o setor internamente.

In [19]:
df_box = df.dropna(subset=['Marketcap_B', 'Sector'])

fig = px.box(df_box,
             x='Sector',
             y='Marketcap_B',
             title='Distribuição de Market Cap por Setor',
             labels={'Marketcap_B': 'Market Cap (USD Bilhões)', 'Sector': 'Setor'},
             color='Sector')
fig.update_layout(xaxis_tickangle=-45, showlegend=False)
fig.show()

### Conclusões

1. Technology é o setor mais desigual de todos

Duas empresas (provavelmente Nvidia ~3800B e Apple ~3300B) estão completamente isoladas acima de tudo. A caixa do box plot em si está quase no zero — ou seja, a maioria das empresas de tech é relativamente pequena, e poucas gigantes dominam.

2. Consumer Cyclical e Communication Services também têm outliers extremos

Provavelmente Amazon (~2400B) em Consumer Cyclical e Alphabet/Google (~2400B) em Communication Services. Mesma dinâmica: uma ou duas empresas distorcem o setor inteiro.

3. Os setores mais homogêneos são Utilities, Real Estate e Basic Materials

As caixas são pequenas e sem outliers extremos, e as empresas desses setores têm tamanhos mais parecidos entre si. São setores maduros, sem "vencedores absolutos".

4. Conclusão geral:

Em quase todo setor do S&P 500 existe uma ou duas empresas que dominam e distorcem a média. Isso significa que análises usando média de market cap por setor são enganosas, a mediana seria mais representativa da empresa típica de cada setor.

## Outliers: Empresas Pequenas Crescendo Rápido

Identificamos empresas com crescimento de receita acima de 50% e market cap abaixo de 
US$100B: o perfil típico de empresas em expansão acelerada dentro do índice S&P500.

In [22]:
# Lista das empresas outliers
outliers = df_outliers[df_outliers['Destaque'] == True][
    ['Shortname', 'Symbol', 'Sector', 'Marketcap_B', 'Revenuegrowth']
].sort_values('Revenuegrowth', ascending=False)

print(f"Total de empresas high growth small cap: {len(outliers)}")
print(outliers.to_string())

# Scatter só dos outliers, colorido por setor
fig = px.scatter(outliers,
                 x='Marketcap_B',
                 y='Revenuegrowth',
                 color='Sector',
                 hover_name='Shortname',
                 hover_data={'Symbol': True, 'Marketcap_B': True, 'Revenuegrowth': True},
                 title='Outliers por Setor: High Growth Small Cap',
                 labels={'Marketcap_B': 'Market Cap (USD Bilhões)',
                         'Revenuegrowth': 'Crescimento de Receita'},
                 size='Marketcap_B',
                 size_max=30)
fig.show()

Total de empresas high growth small cap: 7
                           Shortname Symbol              Sector  Marketcap_B  Revenuegrowth
307             Smurfit WestRock plc     SW   Consumer Cyclical    27.713911          1.632
384       Super Micro Computer, Inc.   SMCI          Technology    18.497999          1.430
223       Prudential Financial, Inc.    PRU  Financial Services    41.939350          1.334
215              Newmont Corporation    NEM     Basic Materials    43.579867          0.847
341  Cincinnati Financial Corporatio   CINF  Financial Services    22.598461          0.833
216      Discover Financial Services    DFS  Financial Services    43.475628          0.605
196                     Vistra Corp.    VST           Utilities    47.614624          0.539


### Conclusões:

Só 7 empresas em 500 atendem aos critérios, o que mostra como crescimento acelerado é raro entre empresas maduras do índice.

Sobre os setores:

Financial Services domina com 3 das 7 empresas (Prudential, Cincinnati Financial, Discover). Isso é curioso dado que excluímos esse setor da análise de valuation por limitações do EBITDA, mas em crescimento de receita eles aparecem forte aqui.

Sobre as empresas individualmente:

SMCI é a mais interessante: pequena (~18B) e crescimento de 140%, impulsionada pela demanda de servidores para IA.
Smurfit WestRock lidera em crescimento (~160%) sendo uma empresa de embalagens, setor inesperado para esse perfil, provavelmente reflexo de uma fusão recente que inflou a receita.
Vistra Corp (Utilities) cresce 50%+, o que é bem incomum para o setor mais estável do índice, provavelmente impulsionada pela demanda energética dos data centers de IA

Conclusão geral:
O crescimento acelerado no S&P 500 não é exclusivo de tech. Empresas de setores "chatos" como utilities e materiais básicos estão crescendo rápido por razões ligadas indiretamente ao boom de IA

## Geografia: Concentração de Market Cap por Estado

O valor de mercado do S&P 500 está distribuído igualmente pelo país?
Analisamos quais estados concentram mais riqueza corporativa.

In [24]:
fig = px.bar(geo_by_state,
             x='State',
             y='Marketcap_B',
             title='Top 15 Estados por Market Cap Total (USD Bilhões)',
             labels={'Marketcap_B': 'Market Cap Total (USD Bilhões)', 'State': 'Estado'},
             color='State',
             color_discrete_sequence=px.colors.qualitative.Plotly)
fig.update_layout(xaxis_tickangle=-45, showlegend=False)
fig.show()

### Conclusões:

1. Califórnia domina de forma absurda

~20T de market cap, praticamente o triplo do segundo colocado. É onde estão Apple, Nvidia, Google, Meta e Netflix. Um estado representa sozinho uma fatia desproporcional do índice inteiro.

2. Washington em segundo surpreende

~6.500B, acima do Texas. Isso é basicamente Microsoft e Amazon, dois gigantes que sozinhos colocam Washington no top 2.

3. Texas e New York empatados em terceiro e quarto

Texas (~4.500B) tem uma base mais diversificada — energia, tech (Tesla) e finanças. New York (~4.500B) é dominado por Financial Services (bancos e seguradoras)

4. Todos os outros estados somados mal chegam perto da Califórnia
Illinois, Ohio, Geórgia e os demais ficam muito abaixo, com barras quase invisíveis na escala do gráfico.

Conclusão geral:
O S&P 500 geograficamente é um índice de 4 estados na prática: Califórnia, Washington, Texas e New York concentram a esmagadora maioria do valor de mercado. O restante do país tem presença simbólica no índice.